In [1]:
from langchain_core.documents import Document  # LangChain Document 클래스 import

# 더미 벡터DB에서 경제 관련 Document 리스트를 반환하는 함수
def retrieve_vectordb(query=None):
    return [  # Document 객체 리스트 반환
        Document(  # 1번 문서 생성
            page_content="""  
정부는 2026년을 '한국경제 대도약 원년'으로 선포하고, 0%대까지 추락했던 잠재성장률을 다시 1.8% 이상으로 끌어올리기 위한 적극적 확장 재정 정책을 추진하고 있습니다.
기존의 감세 중심에서 '지출 중심'으로 전환하며, 내년도 성장률 목표 달성을 위한 대규모 예산 투입을 공식화했습니다.
규제 측면에서는 금산분리 완화 등 실용주의적 개혁을 추진하며, 한국은행은 2026년 1월 기준금리를 연 2.50%로 동결하며 신중한 기조를 유지 중입니다.
""",  # 문서 본문(정책 방향)
            metadata={  # 문서 메타데이터
                "source": "DOC1",  # 문서 ID/출처
                "title": "2026년 경제정책 방향",  # 문서 제목
                "category": "Policy"  # 문서 분류
            }
        ),
        Document(  # 2번 문서 생성
            page_content="""  
2026년 한국 경제는 수출 증가세가 둔화되는 가운데 내수가 완만한 회복세를 보이며 1.8% 수준의 성장을 기록할 것으로 전망됩니다.
이는 2025년의 저성장(0.7~0.8%)에서 벗어나 정상 궤도로 진입하는 과정입니다.
반도체, AI, 조선, 방산 분야는 긍정적이나 미·중 통상 갈등 및 지정학적 리스크로 인한 수출 불확실성은 여전히 높은 상황입니다.
""",  # 문서 본문(전망)
            metadata={  # 문서 메타데이터
                "source": "DOC2",  # 문서 ID/출처
                "title": "2026년 경제 전망",  # 문서 제목
                "category": "Forecast"  # 문서 분류
            }
        ),
        Document(  # 3번 문서 생성
            page_content="""  
2025년의 저성장 충격을 극복하기 위해 정부는 양극화 구조 타파와 지속 가능한 성장에 집중하고 있습니다.
대·중소기업 상생, 지역 균형 발전, 노동시장 이중구조 완화를 핵심 과제로 설정하였습니다.
또한 탄소중립, 에너지 전환 등 ESG 가치를 반영한 R&D 혁신과 AI 대전환을 통해 장기적인 국가 경쟁력 확보에 주력하고 있습니다.
""",  # 문서 본문(전략)
            metadata={  # 문서 메타데이터
                "source": "DOC3",  # 문서 ID/출처
                "title": "경기 침체 탈출 및 양극화 해소",  # 문서 제목
                "category": "Strategy"  # 문서 분류
            }
        )
    ]

# 데이터 확인
docs = retrieve_vectordb()  # 문서 리스트 로드(더미 벡터DB 조회)

In [2]:
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field # 출력 스키마 검증/구조화
from typing import List 
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

class ExpertOpinion(BaseModel):
    role: str = Field(description='전문가 역할')
    analysis: str = Field(descroption = '해당 전문가의 상세 분석내용')
    verdict: Literal['긍정, '부정','중립'] = Field(description = '분석 결과에 대한 종합 평가')
     
class FinalOpinion(BaseModel):
    opinions: List[ExpertOpinion] = Field(description = '개별 전문가들의 의견 리스트')
    final_analysis: str = Field(description = '전문가의 의견을 종합한 최종 분석 내용')
    final_verdict: Literal['긍정','부정','중립'] = Field(description = '전문가의 의견을 종합한 최종 평가')

llm = init_chat_model('gpt-5.6-luna')
prompt = PromptTemplate.from_template('''
당신은 현실세계의 복잡한 경제문제를 해결하기 위해 다양한 관점으로 바라보고, 분석하는 에이젼트입니다.
다음 세가지 관점에서 주어진 문서를 분석하고, 이를 종합하여 결론을 작성해주세요.

1. 분석가1 (국제거시경제): 글로벌 트렌드와 거시지표(성장률, 금리) 관점에서 분석 전문가 (긍정/중립/부정)
2. 분석가2 (지역경제/내수): 내수 소비, 건설, 지역경제활성화 관점에서 분석 전문가 (긍정/중립/부정)
3. 분석가3 (ESG/지속가능성): 장기적 안정성과 환경/사회적 영향 관점에서 분석 전문가 (긍정/중립/부정)

위 분석가들의 의견을 토대로 최종결론을 도출해주세요.  (긍정/중립/부정)

[문서]
{context}

[사용자 질문]
{query}

[조회된 문서]
{docs}

[사용자 질문]
{query}
''')

chain = prompt | llm.with_structured_output(FinalOpinion) # 응답을 FinalOpinion 스키마로 강제

context = '\n\n'.join([doc.page_content for doc in retrieve_vectordb()])
query = '2026년 현재 한국 정부의 경제 정책 예상은?'
final_opinion = chain.invoke({'docs': context, 'query':query})
print(final_option)



SyntaxError: unterminated string literal (detected at line 10) (3363203026.py, line 10)

In [ ]:
for opinion in final_opinion.opinions::
    print(f"[{opinion.role} 의견 : {opinion.verdict}]")
    print(opinion.analysis)
    print()

print("="*100)

print(f"[최종 의견 {final_opinion.final_verdict}]")
print(final_opinion.final_analysis)